# LeetCode 934 — Shortest Bridge bằng Python

Link đề: https://leetcode.com/problems/shortest-bridge/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Depth-First Search (DFS), Breadth-First Search (BFS), Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def shortestBridge(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: lưới có **đúng hai hòn đảo** (nhóm ô `1` liên thông 4 hướng). Ta cần **lật ít ô `0` thành `1` nhất** để nối hai đảo. Cách giải kinh điển gồm **hai giai đoạn**: (1) **DFS/BFS** để tìm toàn bộ ô của **đảo thứ nhất**; (2) **BFS đa nguồn** bắt đầu từ **mọi ô** của đảo đó, lan rộng theo từng lớp qua các ô `0` cho tới khi chạm đảo thứ hai: số lớp đã lan là số ô cần lật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưới nhị phân có đúng hai đảo; tìm số ô nước ít nhất phải lấp (lật thành đất) để hai đảo thành một.
2. Nắm các công cụ Python cần thiết: `collections.deque`, mảng hướng `(-1, 0, 1, 0, -1)` với `pairwise`/`zip`, DFS bằng ngăn xếp tường minh, và đánh dấu đã thăm bằng một giá trị đặc biệt (`2`) trong chính lưới.
3. Hiểu nguyên lý: **BFS đa nguồn** từ một **tập** ô cho khoảng cách ngắn nhất từ tập đó tới mọi ô; số ô nước trên cầu ngắn nhất bằng **khoảng cách (số bước) trừ `1`**; ở đây ta đếm theo lớp để ra đúng số ô nước.
4. Đi qua 4 cách: lấy khoảng cách Manhattan nhỏ nhất giữa mọi cặp ô hai đảo (brute độc lập), DFS tìm đảo rồi BFS đa nguồn theo lớp (cách nộp chính), hai lần BFS lấy tổng khoảng cách, và BFS `0-1` từ một ô duy nhất; đối chiếu bằng ba oracle.
5. Tránh các lỗi kinh điển: chỉ BFS từ một ô của đảo, đếm lệch `1`, cho phép đi chéo, lấy khoảng cách của một cặp ô tùy ý, và DFS đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `grid` cỡ `n x n`: `1` là đất, `0` là nước.

**Đảo** là một nhóm các ô `1` liên thông theo **4 hướng** (trên, dưới, trái, phải) và không liên thông với ô `1` nào khác. Trong `grid` có **đúng hai đảo**.

Bạn có thể đổi một số ô `0` thành `1` để nối hai đảo thành **một đảo**.

Trả về **số ô `0` ít nhất** cần đổi để nối hai đảo.

**Ví dụ 1:** `grid = [[0,1],[1,0]]` -> `1`.

**Ví dụ 2:** `grid = [[0,1,0],[0,0,0],[0,0,1]]` -> `2`.

**Ví dụ 3:** `grid = [[1,1,1,1,1],[1,0,0,0,1],[1,0,1,0,1],[1,0,0,0,1],[1,1,1,1,1]]` -> `1` (đảo trong và đảo ngoài).

**Ràng buộc:**

- `n == grid.length == grid[i].length`
- `2 <= n <= 100`
- `grid[i][j]` là `0` hoặc `1`.
- Có **đúng hai** đảo trong `grid`.

In [1]:
from typing import List, Tuple
from collections import deque
from itertools import combinations, pairwise
import random, time, sys

examples = [
    ([[0, 1], [1, 0]], 1),
    ([[0, 1, 0], [0, 0, 0], [0, 0, 1]], 2),
    ([[1, 1, 1, 1, 1], [1, 0, 0, 0, 1], [1, 0, 1, 0, 1], [1, 0, 0, 0, 1], [1, 1, 1, 1, 1]], 1),
]
for g, expected in examples:
    print(f"n={len(g)} kỳ vọng={expected}")

n=2 kỳ vọng=1
n=3 kỳ vọng=2
n=5 kỳ vọng=1


## 3. Kiến thức nền

### 3.1. Tìm các đảo bằng DFS (ngăn xếp tường minh)

Duyệt lưới; gặp ô `1` chưa thăm thì chạy DFS/BFS để gom mọi ô liên thông 4 hướng. Dùng **ngăn xếp tường minh** thay vì đệ quy để tránh tràn giới hạn đệ quy (một đảo có thể có tới `10^4` ô).

In [2]:
def components(grid: List[List[int]]) -> List[List[Tuple[int, int]]]:
    n = len(grid)
    seen = [[False] * n for _ in range(n)]
    comps = []
    for i in range(n):
        for j in range(n):
            if grid[i][j] == 1 and not seen[i][j]:
                comp, stack = [], [(i, j)]
                seen[i][j] = True
                while stack:
                    x, y = stack.pop()
                    comp.append((x, y))
                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < n and 0 <= ny < n and grid[nx][ny] == 1 and not seen[nx][ny]:
                            seen[nx][ny] = True
                            stack.append((nx, ny))
                comps.append(comp)
    return comps

for g, _ in examples:
    print([len(c) for c in components(g)], "ô mỗi đảo")

[1, 1] ô mỗi đảo
[1, 1] ô mỗi đảo
[16, 1] ô mỗi đảo


### 3.2. BFS đa nguồn theo lớp

Đặt **mọi ô của đảo thứ nhất** vào hàng đợi cùng lúc. Mỗi lần xử lý **một lớp** (`len(queue)` phần tử hiện có): các ô nước kề được thêm vào lớp kế (đánh dấu đã thăm). Khi một ô kề là **đất của đảo kia**, số lớp đã hoàn thành chính là số ô nước đã vượt qua.

In [3]:
q = deque([1, 2, 3])
layer = 0
while q:
    for _ in range(len(q)):          # duyệt đúng số phần tử của lớp hiện tại
        x = q.popleft()
        if x < 10:
            q.append(x * 3)
    layer += 1
    print("lớp", layer, "xong, hàng đợi còn", list(q))

lớp 1 xong, hàng đợi còn [3, 6, 9]
lớp 2 xong, hàng đợi còn [9, 18, 27]
lớp 3 xong, hàng đợi còn [27]
lớp 4 xong, hàng đợi còn []


### 3.3. Mảng hướng và 4-liên thông

`dirs = (-1, 0, 1, 0, -1)`; `pairwise(dirs)` sinh các cặp `(-1, 0), (0, 1), (1, 0), (0, -1)`: lên, phải, xuống, trái. Đề dùng liên thông **4 hướng**; đi chéo là sai.

In [4]:
print(list(pairwise((-1, 0, 1, 0, -1))))

[(-1, 0), (0, 1), (1, 0), (0, -1)]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `[[0,1,0],[0,0,0],[0,0,1]]`. Đảo A là `(0,1)`, đảo B là `(2,2)`.

- Lớp 0 (khoảng cách `0`): nguồn `(0,1)`.
- Lớp 1 (khoảng cách `1`): các ô nước kề `(0,1)`: `(0,0)`, `(0,2)`, `(1,1)`.
- Lớp 2 (khoảng cách `2`): các ô nước kề các ô trên: `(1,0)`, `(1,2)`, ... Ô `(1,2)` kề `(2,2)` thuộc đảo B.

Đường ngắn nhất `(0,1) -> (0,2) -> (1,2) -> (2,2)` (hoặc qua `(1,1)`) có `3` cạnh, trong đó **hai ô nước** `(0,2)` và `(1,2)` cần lật. Đáp án `2` = số cạnh `- 1`.

## 5. Cách 1 — Brute Force: khoảng cách Manhattan nhỏ nhất giữa hai đảo

**Ý tưởng.** Tìm hai đảo (hai danh sách ô `A`, `B`). Với mọi cặp `a in A`, `b in B`, tính khoảng cách Manhattan `|ax - bx| + |ay - by|`; lấy nhỏ nhất `d`. Đáp án `d - 1`.

**Vì sao đúng (tóm tắt, chứng minh ở mục 10).** Với cặp gần nhất, mọi ô nằm trên đường đơn điệu giữa chúng là nước (nếu có ô đất của một đảo trên đường thì có cặp gần hơn), và đường đơn điệu nằm trong lưới. Vậy cần lật đúng `d - 1` ô. Ngược lại mọi cầu có `k` ô nước nối một ô `a` với một ô `b` bằng đường 4-liên thông độ dài `k + 1 >= |a - b|` (Manhattan). Đây là một oracle **hoàn toàn khác** với BFS.

In [5]:
def bridge_manhattan(grid: List[List[int]]) -> int:
    A, B = components(grid)
    d = min(abs(ax - bx) + abs(ay - by) for ax, ay in A for bx, by in B)
    return d - 1

for g, expected in examples:
    got = bridge_manhattan(g)
    print(f"n={len(g)} ->", got, "OK" if got == expected else "SAI")
    assert got == expected

n=2 -> 1 OK
n=3 -> 2 OK
n=5 -> 1 OK


**Độ phức tạp và nút thắt.** `O(|A| * |B|)` với tối đa `~ (n^2 / 2)^2 = 2,5 * 10^7` cặp khi `n = 100`: chậm trong Python. Xem thời gian khi hai đảo lớn:

In [6]:
def two_big_islands(n):
    """Hai đảo: cột 0 (đất) và các ô nửa phải; cách nhau một vùng nước."""
    g = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n // 3):
            g[i][j] = 1
        for j in range(2 * n // 3 + 2, n):
            g[i][j] = 1
    return g

for n in (10, 20, 30, 40, 60):
    g = two_big_islands(n)
    t0 = time.perf_counter()
    r = bridge_manhattan(g)
    print(f"n={n:3}: kết quả {r:3}, {1000 * (time.perf_counter() - t0):9.1f} ms")

n= 10: kết quả   5,       0.2 ms
n= 20: kết quả   9,       1.7 ms
n= 30: kết quả  12,      10.1 ms
n= 40: kết quả  15,      19.6 ms
n= 60: kết quả  22,      95.9 ms


Thời gian tăng với **tích số ô hai đảo**. Nút thắt: ta xét mọi cặp thay vì chỉ khám phá dần từ một đảo ra ngoài. BFS đa nguồn làm điều đó trong `O(n^2)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Lan dần từ đảo thứ nhất theo từng lớp cho tới khi chạm đảo thứ hai.** Mỗi lớp lan thêm một vòng ô nước quanh vùng đã lan; số lớp cho tới khi chạm đảo thứ hai chính là số ô nước ít nhất cần lấp, vì BFS đa nguồn cho khoảng cách ngắn nhất từ **tập nguồn** tới mọi ô.

| Cách | Ý tưởng |
|---|---|
| 2 | DFS lấy đảo thứ nhất, BFS đa nguồn theo lớp (nộp) |
| 3 | Hai lần BFS đa nguồn (từ mỗi đảo), đáp án `min(dA + dB) - 1` |
| 4 | BFS `0-1` từ **một** ô: đi vào ô đất giá `0`, ô nước giá `1` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS lấy đảo thứ nhất, BFS đa nguồn theo lớp (cách nộp chính)

**Giải thích từng đoạn.**

- Tìm ô `1` đầu tiên (duyệt lưới theo hàng); DFS (ngăn xếp tường minh) gom mọi ô của đảo đó và **đánh dấu `2`** trong lưới để phân biệt với đảo kia.
- Đưa mọi ô đó vào `deque` và BFS theo lớp: với mỗi ô lấy ra, xét 4 láng giềng: ô `1` (đảo kia) ⇒ trả `steps`; ô `0` ⇒ đánh dấu `2` và thêm vào hàng đợi.
- Sau mỗi lớp `steps += 1`.
- Số `steps` khi gặp đảo kia chính là số ô nước đã lấp (lớp `0` là chính đảo thứ nhất, nên lớp `k` lan tới các ô nước ở khoảng cách `k`; ô đất đảo kia nằm ở khoảng cách `steps + 1` nên số ô nước giữa là `steps`).

In [7]:
def bridge_bfs(grid: List[List[int]]) -> int:
    g = [row[:] for row in grid]
    n = len(g)
    start = next((i, j) for i in range(n) for j in range(n) if g[i][j] == 1)
    island, stack = [], [start]
    g[start[0]][start[1]] = 2
    while stack:
        x, y = stack.pop()
        island.append((x, y))
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and g[nx][ny] == 1:
                g[nx][ny] = 2
                stack.append((nx, ny))
    q = deque(island)
    steps = 0
    while q:
        for _ in range(len(q)):
            x, y = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n:
                    if g[nx][ny] == 1:
                        return steps
                    if g[nx][ny] == 0:
                        g[nx][ny] = 2
                        q.append((nx, ny))
        steps += 1
    return -1

for g, expected in examples:
    got = bridge_bfs(g)
    print(f"n={len(g)} ->", got, "OK" if got == expected else "SAI")
    assert got == expected

n=2 -> 1 OK
n=3 -> 2 OK
n=5 -> 1 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def shortestBridge(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        dirs = (-1, 0, 1, 0, -1)
        start = next((i, j) for i in range(n) for j in range(n) if grid[i][j])
        island, stack = [], [start]
        grid[start[0]][start[1]] = 2
        while stack:
            x, y = stack.pop()
            island.append((x, y))
            for a, b in zip(dirs, dirs[1:]):
                nx, ny = x + a, y + b
                if 0 <= nx < n and 0 <= ny < n and grid[nx][ny] == 1:
                    grid[nx][ny] = 2
                    stack.append((nx, ny))
        q = deque(island)
        steps = 0
        while q:
            for _ in range(len(q)):
                x, y = q.popleft()
                for a, b in zip(dirs, dirs[1:]):
                    nx, ny = x + a, y + b
                    if 0 <= nx < n and 0 <= ny < n:
                        if grid[nx][ny] == 1:
                            return steps
                        if grid[nx][ny] == 0:
                            grid[nx][ny] = 2
                            q.append((nx, ny))
            steps += 1
        return -1

sol_bfs = Solution().shortestBridge
assert [sol_bfs([r[:] for r in g]) for g, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n^2)` thời gian (mỗi ô vào hàng đợi tối đa một lần), `O(n^2)` bộ nhớ. Lưu ý: hàm sửa trực tiếp `grid` đầu vào (đánh dấu `2`).

---

### 7.2. Cách 3 — Hai lần BFS đa nguồn

**Ý tưởng khác.** Tính `dA[c]` = khoảng cách (số bước 4 hướng) từ **tập** ô đảo `A` tới ô `c`, và `dB[c]` tương tự cho `B`; BFS thường (không giới hạn trong nước). Một cầu `A -> ... -> B` với `k` ô nước có `k + 1` bước, đi qua mỗi ô `c` trên đường với `dA[c] + dB[c] = k + 1` (đường ngắn nhất). Vậy đáp án `min_c (dA[c] + dB[c]) - 1`.

**Giải thích từng đoạn.** BFS đa nguồn: đưa mọi ô của đảo vào hàng đợi với khoảng cách `0`; mở rộng qua **mọi** ô của lưới (cho phép đi qua cả ô đất của đảo kia).

In [9]:
def multi_source_dist(n: int, sources) -> List[List[int]]:
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    q = deque()
    for x, y in sources:
        dist[x][y] = 0
        q.append((x, y))
    while q:
        x, y = q.popleft()
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and dist[nx][ny] == INF:
                dist[nx][ny] = dist[x][y] + 1
                q.append((nx, ny))
    return dist

def bridge_two_bfs(grid: List[List[int]]) -> int:
    n = len(grid)
    A, B = components(grid)
    dA, dB = multi_source_dist(n, A), multi_source_dist(n, B)
    return min(dA[i][j] + dB[i][j] for i in range(n) for j in range(n)) - 1

for g, expected in examples:
    got = bridge_two_bfs(g)
    print(f"n={len(g)} ->", got, "OK" if got == expected else "SAI")
    assert got == expected

n=2 -> 1 OK
n=3 -> 2 OK
n=5 -> 1 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def shortestBridge(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        seen = [[0] * n for _ in range(n)]
        islands = []
        for i in range(n):
            for j in range(n):
                if grid[i][j] == 1 and not seen[i][j]:
                    comp, stack = [], [(i, j)]
                    seen[i][j] = 1
                    while stack:
                        x, y = stack.pop()
                        comp.append((x, y))
                        for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                            if 0 <= nx < n and 0 <= ny < n and grid[nx][ny] == 1 and not seen[nx][ny]:
                                seen[nx][ny] = 1
                                stack.append((nx, ny))
                    islands.append(comp)

        def bfs(sources):
            dist = [[-1] * n for _ in range(n)]
            q = deque(sources)
            for x, y in sources:
                dist[x][y] = 0
            while q:
                x, y = q.popleft()
                for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                    if 0 <= nx < n and 0 <= ny < n and dist[nx][ny] < 0:
                        dist[nx][ny] = dist[x][y] + 1
                        q.append((nx, ny))
            return dist

        da, db = bfs(islands[0]), bfs(islands[1])
        return min(da[i][j] + db[i][j] for i in range(n) for j in range(n)) - 1

sol_two = Solution().shortestBridge
assert [sol_two([r[:] for r in g]) for g, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n^2)` thời gian (ba lượt duyệt), `O(n^2)` bộ nhớ.

---

### 7.3. Cách 4 — BFS `0-1` từ một ô duy nhất

**Ý tưởng khác.** Không cần liệt kê đảo trước: chọn **một ô đất bất kỳ** làm nguồn, chạy BFS `0-1` (hàng đợi hai đầu): bước vào ô **đất** tốn `0`, bước vào ô **nước** tốn `1`. Khi đó mọi ô của đảo chứa nguồn có khoảng cách `0` (di chuyển trong đảo miễn phí), và mỗi ô đất của **đảo kia** có khoảng cách bằng số ô nước phải đi qua. Đáp án là khoảng cách nhỏ nhất **dương** của một ô đất (ô đất có khoảng cách `0` thuộc đảo nguồn).

**Giải thích từng đoạn.** Dùng `deque`: cạnh trọng số `0` thì `appendleft`, trọng số `1` thì `append`; cập nhật khoảng cách khi tìm được đường ngắn hơn (có thể vào hàng đợi nhiều lần).

In [11]:
def bridge_01_bfs(grid: List[List[int]]) -> int:
    n = len(grid)
    start = next((i, j) for i in range(n) for j in range(n) if grid[i][j] == 1)
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    dist[start[0]][start[1]] = 0
    dq = deque([start])
    while dq:
        x, y = dq.popleft()
        d = dist[x][y]
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                w = 0 if grid[nx][ny] == 1 else 1
                if d + w < dist[nx][ny]:
                    dist[nx][ny] = d + w
                    if w == 0:
                        dq.appendleft((nx, ny))
                    else:
                        dq.append((nx, ny))
    return min(dist[i][j] for i in range(n) for j in range(n) if grid[i][j] == 1 and dist[i][j] > 0)

for g, expected in examples:
    got = bridge_01_bfs(g)
    print(f"n={len(g)} ->", got, "OK" if got == expected else "SAI")
    assert got == expected

n=2 -> 1 OK
n=3 -> 2 OK
n=5 -> 1 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def shortestBridge(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        start = next((i, j) for i in range(n) for j in range(n) if grid[i][j])
        INF = n * n + 5
        dist = [[INF] * n for _ in range(n)]
        dist[start[0]][start[1]] = 0
        dq = deque([start])
        while dq:
            x, y = dq.popleft()
            d = dist[x][y]
            for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                if 0 <= nx < n and 0 <= ny < n:
                    w = 0 if grid[nx][ny] else 1
                    if d + w < dist[nx][ny]:
                        dist[nx][ny] = d + w
                        if w:
                            dq.append((nx, ny))
                        else:
                            dq.appendleft((nx, ny))
        return min(dist[i][j] for i in range(n) for j in range(n) if grid[i][j] and dist[i][j] > 0)

sol_01 = Solution().shortestBridge
assert [sol_01([r[:] for r in g]) for g, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n^2)` thời gian (mỗi ô được cập nhật `O(1)` lần với BFS `0-1`), `O(n^2)` bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 2 (BFS theo lớp) cho ví dụ 2

In [13]:
def trace_bfs(grid):
    g = [row[:] for row in grid]
    n = len(g)
    A = components(grid)[0]
    for x, y in A:
        g[x][y] = 2
    q = deque(A)
    steps = 0
    print("lớp 0 (đảo thứ nhất):", list(q))
    while q:
        layer = len(q)
        nxt = []
        for _ in range(layer):
            x, y = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n:
                    if g[nx][ny] == 1:
                        print(f"ô ({x},{y}) kề đảo kia tại ({nx},{ny}): trả về steps = {steps}")
                        return steps
                    if g[nx][ny] == 0:
                        g[nx][ny] = 2
                        q.append((nx, ny))
                        nxt.append((nx, ny))
        steps += 1
        print(f"lớp {steps}: {nxt}")
    return -1

assert trace_bfs(examples[1][0]) == 2

lớp 0 (đảo thứ nhất): [(0, 1)]
lớp 1: [(1, 1), (0, 2), (0, 0)]
lớp 2: [(2, 1), (1, 2), (1, 0)]
ô (2,1) kề đảo kia tại (2,2): trả về steps = 2


Cách đọc: `steps` bắt đầu bằng `0` khi ta xử lý các ô của đảo thứ nhất (lớp 0); sau khi xử lý lớp 0, `steps = 1` và ta xử lý các ô nước ở khoảng cách `1`. Khi một ô đang xử lý ở lớp có `steps = k` thấy ô đất của đảo kia ở bên cạnh thì ô đó cách đảo thứ nhất `k` bước, tức là có `k` ô nước (các ô ở khoảng cách `1..k`) nằm trên cầu; vì vậy trả `steps`. Với ví dụ 2 ta chạm ở `steps = 2`.

### 8.2. Cách 3 (hai BFS) cho ví dụ 1: `[[0,1],[1,0]]`

`A = {(0,1)}`, `B = {(1,0)}`. `dA = [[1,0],[2,1]]`, `dB = [[1,2],[0,1]]`. Tổng: `[[2,2],[2,2]]`; nhỏ nhất `2`; đáp án `2 - 1 = 1`.

In [14]:
g = examples[0][0]
A, B = components(g)
dA, dB = multi_source_dist(2, A), multi_source_dist(2, B)
print("dA =", dA, "| dB =", dB, "| tổng =", [[dA[i][j] + dB[i][j] for j in range(2)] for i in range(2)])
assert min(dA[i][j] + dB[i][j] for i in range(2) for j in range(2)) - 1 == 1

dA = [[1, 0], [2, 1]] | dB = [[1, 2], [0, 1]] | tổng = [[2, 2], [2, 2]]


### 8.3. Cách 4 (BFS `0-1`) cho ví dụ 2

Nguồn `(0, 1)` (khoảng cách `0`). Các ô nước kề có khoảng cách `1` rồi `2`; khi tới ô đất `(2, 2)` (bước vào ô đất tốn `0`) khoảng cách là `2` (đi qua hai ô nước): đáp án `2`.

In [15]:
def trace_01(grid):
    n = len(grid)
    start = next((i, j) for i in range(n) for j in range(n) if grid[i][j] == 1)
    INF = 10 ** 9
    dist = [[INF] * n for _ in range(n)]
    dist[start[0]][start[1]] = 0
    dq = deque([start])
    while dq:
        x, y = dq.popleft()
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                w = 0 if grid[nx][ny] == 1 else 1
                if dist[x][y] + w < dist[nx][ny]:
                    dist[nx][ny] = dist[x][y] + w
                    (dq.appendleft if w == 0 else dq.append)((nx, ny))
    for row in dist:
        print(row)
    return dist

d = trace_01(examples[1][0])
assert d[2][2] == 2

[1, 0, 1]
[2, 1, 2]
[3, 2, 2]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (DFS lấy đảo thứ nhất rồi BFS đa nguồn theo lớp): cách kinh điển, `O(n^2)`.

```python
class Solution:
    def shortestBridge(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        dirs = (-1, 0, 1, 0, -1)
        start = next((i, j) for i in range(n) for j in range(n) if grid[i][j])
        island, stack = [], [start]
        grid[start[0]][start[1]] = 2
        while stack:
            x, y = stack.pop()
            island.append((x, y))
            for a, b in zip(dirs, dirs[1:]):
                nx, ny = x + a, y + b
                if 0 <= nx < n and 0 <= ny < n and grid[nx][ny] == 1:
                    grid[nx][ny] = 2
                    stack.append((nx, ny))
        q = deque(island)
        steps = 0
        while q:
            for _ in range(len(q)):
                x, y = q.popleft()
                for a, b in zip(dirs, dirs[1:]):
                    nx, ny = x + a, y + b
                    if 0 <= nx < n and 0 <= ny < n:
                        if grid[nx][ny] == 1:
                            return steps
                        if grid[nx][ny] == 0:
                            grid[nx][ny] = 2
                            q.append((nx, ny))
            steps += 1
        return -1
```

In [16]:
class Solution:
    def shortestBridge(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        dirs = (-1, 0, 1, 0, -1)
        start = next((i, j) for i in range(n) for j in range(n) if grid[i][j])
        island, stack = [], [start]
        grid[start[0]][start[1]] = 2
        while stack:
            x, y = stack.pop()
            island.append((x, y))
            for a, b in zip(dirs, dirs[1:]):
                nx, ny = x + a, y + b
                if 0 <= nx < n and 0 <= ny < n and grid[nx][ny] == 1:
                    grid[nx][ny] = 2
                    stack.append((nx, ny))
        q = deque(island)
        steps = 0
        while q:
            for _ in range(len(q)):
                x, y = q.popleft()
                for a, b in zip(dirs, dirs[1:]):
                    nx, ny = x + a, y + b
                    if 0 <= nx < n and 0 <= ny < n:
                        if grid[nx][ny] == 1:
                            return steps
                        if grid[nx][ny] == 0:
                            grid[nx][ny] = 2
                            q.append((nx, ny))
            steps += 1
        return -1

sol = Solution()
for g, expected in examples:
    got = sol.shortestBridge([r[:] for r in g])
    print(f"n={len(g)} ->", got, "OK" if got == expected else "SAI")
    assert got == expected

n=2 -> 1 OK
n=3 -> 2 OK
n=5 -> 1 OK


## 10. Vì sao đúng

**(a) Mô hình.** Lật tập `S` các ô nước để hai đảo thành **một** đảo nghĩa là `A ∪ B ∪ S` liên thông 4 hướng. Một tập tối thiểu liên thông hai đảo chứa một **đường 4-liên thông** từ một ô `a in A` tới một ô `b in B` đi qua các ô của `S` (mọi ô trung gian là nước, vì chỉ có hai đảo). Số ô nước tối thiểu bằng độ dài ngắn nhất (số ô nước trung gian) của một đường như vậy.

**(b) BFS đa nguồn theo lớp (Cách 2).** Hàng đợi ban đầu chứa mọi ô của `A` (khoảng cách `0`). BFS cho khoảng cách ngắn nhất từ **tập** `A` tới mọi ô, **chỉ đi qua ô nước** (ta chỉ thêm ô `0`); khi một ô ở khoảng cách `k` (nước hoặc chính `A` lúc `k = 0`) kề một ô của `B`, đường từ `A` tới `B` có `k` ô nước trung gian (các ô nước ở khoảng cách `1..k`). Lớp được duyệt theo thứ tự khoảng cách nên lần đầu gặp `B` cho giá trị nhỏ nhất. Biến `steps` tại thời điểm xử lý lớp có khoảng cách `k` bằng `k`: với `k = 0` (ô của `A` kề ngay `B`) đáp án `0` (hai đảo kề nhau; không xảy ra theo đề vì khi đó chỉ có một đảo).

**(c) Cách 3 (hai BFS).** Với mỗi ô `c`, `dA[c] + dB[c]` là độ dài (số cạnh) của đường ngắn nhất `A -> c -> B`; đường tối ưu có `k + 1` cạnh với `k` ô nước nên `min_c (dA[c] + dB[c]) = k + 1` (đạt tại mọi ô trên đường tối ưu), suy ra `k = min - 1`. Khoảng cách BFS cho phép đi qua ô đất của đảo kia nhưng không làm giảm tổng dưới độ dài đường ngắn nhất `A -> B`, vì một ô `c` bất kỳ có `dA[c] + dB[c] >=` độ dài đường ngắn nhất từ `A` tới `B` (bất đẳng thức tam giác).

**(d) Cách 4 (BFS `0-1`).** Đồ thị có trọng số cạnh `0` (vào ô đất) hoặc `1` (vào ô nước); khoảng cách ngắn nhất từ nguồn tới ô `c` là số ô nước tối thiểu trên một đường từ nguồn tới `c`. Mọi ô của đảo chứa nguồn có khoảng cách `0` (đi trong đảo toàn ô đất). Ô đất của đảo kia có khoảng cách `>= 1` (hai đảo không kề nhau) bằng số ô nước tối thiểu của một đường nối: lấy nhỏ nhất ra đáp án.

**(e) Cách 1 (Manhattan).** Đặt `d` là khoảng cách Manhattan nhỏ nhất giữa một ô của `A` và một ô của `B`, tại cặp `(a, b)`. (i) Cận trên: đường đơn điệu từ `a` tới `b` (đi đủ số bước ngang rồi dọc) nằm trong lưới (hình chữ nhật giữa hai ô); mọi ô trung gian là nước, vì nếu có một ô đất của `A` thì nó cùng `b` tạo cặp gần hơn `d`, nếu của `B` thì nó cùng `a` tạo cặp gần hơn. Vậy lật `d - 1` ô là đủ. (ii) Cận dưới: một cầu tối ưu có `k` ô nước là đường 4-liên thông `a' -> ... -> b'` có `k + 1` cạnh, mà số cạnh của mọi đường 4 hướng giữa hai ô không nhỏ hơn khoảng cách Manhattan, nên `k + 1 >= |a' - b'| >= d`.

**Kiểm chứng bằng máy.** Ba oracle **độc lập** với Cách 2: (1) **Manhattan nhỏ nhất** giữa mọi cặp ô (Cách 1); (2) với `n <= 4`, **định nghĩa gốc**: thử mọi tập con ô nước theo kích thước tăng dần, kiểm tra việc lật chúng làm lưới thành **đúng một** đảo; (3) so với hai BFS và BFS `0-1`. Lưới kiểm thử luôn có **đúng hai đảo** (kiểm tra bằng `components`).

In [17]:
def oracle_subsets(grid: List[List[int]]) -> int:
    """Định nghĩa gốc: thử mọi tập con ô nước theo kích thước tăng dần cho tới khi lưới có đúng một đảo."""
    n = len(grid)
    water = [(i, j) for i in range(n) for j in range(n) if grid[i][j] == 0]
    for k in range(0, len(water) + 1):
        for chosen in combinations(water, k):
            g = [row[:] for row in grid]
            for x, y in chosen:
                g[x][y] = 1
            if len(components(g)) == 1:
                return k
    return -1

def random_two_island_grid(rng, n, density):
    """Sinh lưới n x n có đúng hai đảo: mọc hai khối ngẫu nhiên (khối thứ hai không chạm khối thứ nhất theo 4 hướng),
    rồi kiểm tra lại bằng components (độc lập với các lời giải)."""
    total = max(2, int(density * n * n))
    while True:
        g = [[0] * n for _ in range(n)]
        s1 = (rng.randrange(n), rng.randrange(n))
        blob1 = {s1}
        cells1 = [s1]
        for _ in range(rng.randint(1, max(1, total // 2)) * 3):
            x, y = rng.choice(cells1)
            dx, dy = rng.choice(((1, 0), (-1, 0), (0, 1), (0, -1)))
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and (nx, ny) not in blob1:
                blob1.add((nx, ny))
                cells1.append((nx, ny))
        forbidden = set(blob1)
        for x, y in blob1:
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                forbidden.add((x + dx, y + dy))
        free = [(i, j) for i in range(n) for j in range(n) if (i, j) not in forbidden]
        if not free:
            continue
        s2 = rng.choice(free)
        blob2 = {s2}
        cells2 = [s2]
        for _ in range(rng.randint(1, max(1, total // 2)) * 3):
            x, y = rng.choice(cells2)
            dx, dy = rng.choice(((1, 0), (-1, 0), (0, 1), (0, -1)))
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and (nx, ny) not in forbidden and (nx, ny) not in blob2:
                blob2.add((nx, ny))
                cells2.append((nx, ny))
        for x, y in blob1 | blob2:
            g[x][y] = 1
        if len(components(g)) == 2:
            return g

rng = random.Random(1934)
for _ in range(150):
    n = rng.randint(2, 4)
    g = random_two_island_grid(rng, n, rng.choice([0.15, 0.3, 0.45]))
    assert bridge_manhattan(g) == oracle_subsets(g), g
print("Manhattan nhỏ nhất và định nghĩa gốc (thử mọi tập con ô nước) khớp nhau trên 150 lưới nhỏ")
for g, expected in examples[:2]:
    assert oracle_subsets(g) == expected

Manhattan nhỏ nhất và định nghĩa gốc (thử mọi tập con ô nước) khớp nhau trên 150 lưới nhỏ


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Manhattan mọi cặp | `O(\|A\| * \|B\|)` (tối đa `~ n^4`) | `O(n^2)` |
| 2. DFS + BFS đa nguồn | `O(n^2)` | `O(n^2)` |
| 3. Hai BFS | `O(n^2)` | `O(n^2)` |
| 4. BFS `0-1` | `O(n^2)` | `O(n^2)` |

Đo thử với `n = 100` (cỡ tối đa):

In [18]:
rng = random.Random(11)
n = 100
g_big = two_big_islands(n)
assert len(components(g_big)) == 2
ref = sol_bfs([r[:] for r in g_big])
for name, fn in (("Cách 2 DFS+BFS", sol_bfs), ("Cách 3 hai BFS", sol_two), ("Cách 4 BFS 0-1", sol_01)):
    t0 = time.perf_counter()
    got = fn([r[:] for r in g_big])
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref, name
    print(f"{name:16} n={n}: {dt:8.2f} ms (đáp án {got})")
t0 = time.perf_counter()
got = bridge_manhattan(two_big_islands(40))
print(f"{'Cách 1 (n=40)':16}: {1000 * (time.perf_counter() - t0):8.1f} ms, đáp án {got}")

Cách 2 DFS+BFS   n=100:    12.13 ms (đáp án 35)
Cách 3 hai BFS   n=100:    26.05 ms (đáp án 35)
Cách 4 BFS 0-1   n=100:    11.64 ms (đáp án 35)


Cách 1 (n=40)   :     36.1 ms, đáp án 15


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với các oracle, test ngẫu nhiên có seed (lưới luôn có **đúng hai đảo**) và tính chất.

In [19]:
def solutions_list():
    return {
        "bfs": bridge_bfs,
        "two_bfs": bridge_two_bfs,
        "bfs01": bridge_01_bfs,
        "sol_bfs": lambda g: sol_bfs([r[:] for r in g]),
        "sol_two": lambda g: sol_two([r[:] for r in g]),
        "sol_01": lambda g: sol_01([r[:] for r in g]),
        "final": lambda g: Solution().shortestBridge([r[:] for r in g]),
    }

def run_all(grid, expected=None, subsets_limit=4):
    assert len(components(grid)) == 2, "lưới phải có đúng hai đảo"
    ref = bridge_manhattan(grid)
    if len(grid) <= subsets_limit:
        assert oracle_subsets(grid) == ref, ("oracle tập con lệch", grid)
    for name, fn in solutions_list().items():
        assert fn(grid) == ref, (name, grid)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", grid, expected, ref)
    return ref

# 12.1 ví dụ đề
for g, expected in examples:
    assert run_all(g, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([[1, 0], [0, 1]], 1),                                  # hai đảo chéo nhau: lật một ô
    ([[1, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 1]], 5),     # hai góc đối: Manhattan 6 -> 5
    ([[1, 0, 0, 1], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], 2),     # cùng hàng: cách hai ô
    ([[1, 1, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 1, 1]], 3),     # hai đảo ngang: gần nhất (0,1)-(3,2): 4 -> 3
    ([[0, 1, 0], [0, 0, 0], [0, 0, 1]], 2),                 # ví dụ 2
    ([[1, 1, 1, 1, 1], [1, 0, 0, 0, 1], [1, 0, 1, 0, 1], [1, 0, 0, 0, 1], [1, 1, 1, 1, 1]], 1),   # lồng nhau
    ([[1, 0, 1], [0, 0, 0], [0, 0, 0]], 1),                 # hai ô cách nhau một ô nước
    ([[1, 0, 0, 0, 0], [1, 0, 0, 0, 1], [1, 0, 0, 0, 1], [1, 0, 0, 0, 1], [1, 0, 0, 0, 0]], 3),     # hai cột đối diện cách 3 cột nước
]
for g, expected in edge:
    run_all(g, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK


12.2 edge case: OK 8 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_single_source(grid):                       # BFS chỉ từ MỘT ô của đảo thứ nhất
    g = [row[:] for row in grid]
    n = len(g)
    A = components(grid)[0]
    islandA = set(A)
    start = A[0]
    q = deque([start])
    seen = {start}
    steps = 0
    while q:
        for _ in range(len(q)):
            x, y = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n and (nx, ny) not in seen:
                    if g[nx][ny] == 1 and (nx, ny) not in islandA:
                        return steps
                    if g[nx][ny] == 0 or (nx, ny) in islandA:
                        seen.add((nx, ny))
                        q.append((nx, ny))
        steps += 1
    return -1

def wrong_off_by_one(grid):                          # đếm cả bước vào ô đất đảo kia: steps + 1
    return bridge_bfs(grid) + 1

def wrong_diagonal(grid):                            # cho phép đi chéo (8 hướng)
    g = [row[:] for row in grid]
    n = len(g)
    A = components(grid)[0]
    for x, y in A:
        g[x][y] = 2
    q = deque(A)
    steps = 0
    while q:
        for _ in range(len(q)):
            x, y = q.popleft()
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx == dy == 0:
                        continue
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < n and 0 <= ny < n:
                        if g[nx][ny] == 1:
                            return steps
                        if g[nx][ny] == 0:
                            g[nx][ny] = 2
                            q.append((nx, ny))
        steps += 1
    return -1

def wrong_arbitrary_pair(grid):                      # Manhattan của hai ô đầu tiên của mỗi đảo (không lấy nhỏ nhất)
    A, B = components(grid)
    (ax, ay), (bx, by) = A[0], B[0]
    return abs(ax - bx) + abs(ay - by) - 1

def find_counter(wrong, tries=20000, seed=3):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(3, 6)
        g = random_two_island_grid(rng, n, rng.choice([0.2, 0.3, 0.4]))
        if wrong(g) != bridge_manhattan(g):
            return g
    return None

counters = {}
for name, fn in (("BFS chỉ từ một ô", wrong_single_source), ("đếm lệch 1", wrong_off_by_one),
                 ("cho phép đi chéo", wrong_diagonal), ("Manhattan của cặp tuỳ ý", wrong_arbitrary_pair)):
    g = find_counter(fn)
    assert g is not None, name
    counters[name] = g
    print(f"{name:26} n={len(g)}: sai -> {fn(g)}, đúng -> {bridge_manhattan(g)}")
    run_all(g)
print("12.3 phản ví dụ: OK")

BFS chỉ từ một ô           n=4: sai -> 2, đúng -> 1
đếm lệch 1                 n=4: sai -> 2, đúng -> 1
cho phép đi chéo           n=4: sai -> 0, đúng -> 1
Manhattan của cặp tuỳ ý    n=4: sai -> 2, đúng -> 1
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: n = 2..4, so với định nghĩa gốc (tập con ô nước) và Manhattan
rng = random.Random(4934)
for _ in range(400):
    n = rng.randint(2, 4)
    run_all(random_two_island_grid(rng, n, rng.choice([0.1, 0.25, 0.4, 0.55])))
print("12.4 ngẫu nhiên n <= 4 (400 lưới, gồm oracle tập con): OK")

# 12.5 trung bình: n tới 12 so với Manhattan (lưới lọc ngẫu nhiên có đúng hai đảo)
rng = random.Random(5934)
for _ in range(200):
    n = rng.randint(5, 12)
    run_all(random_two_island_grid(rng, n, rng.choice([0.08, 0.15, 0.25])))
print("12.5 ngẫu nhiên n tới 12 (200 lưới): OK")

# 12.6 sinh hai đảo lớn có cấu trúc (n tới 100): mỗi đảo là hình chữ nhật có đáy; đáp án biết trước
def two_rectangles(n, rng):
    """Hai hình chữ nhật tách rời (không kề nhau); đáp án = khoảng trống nhỏ nhất theo Manhattan - 1."""
    while True:
        r1 = (rng.randint(0, n - 2), rng.randint(0, n - 2))
        h1, w1 = rng.randint(1, max(1, n // 3)), rng.randint(1, max(1, n // 3))
        r2 = (rng.randint(0, n - 2), rng.randint(0, n - 2))
        h2, w2 = rng.randint(1, max(1, n // 3)), rng.randint(1, max(1, n // 3))
        g = [[0] * n for _ in range(n)]
        for (r, c), h, w, val in ((r1, h1, w1, 1), (r2, h2, w2, 1)):
            for i in range(r, min(n, r + h)):
                for j in range(c, min(n, c + w)):
                    g[i][j] = val
        if len(components(g)) == 2:
            return g

rng = random.Random(6934)
for n in (20, 40, 70, 100):
    for _ in range(6):
        g = two_rectangles(n, rng)
        ref = bridge_manhattan(g)
        for name, fn in solutions_list().items():
            assert fn(g) == ref, (n, name)
print("12.6 hai hình chữ nhật, n tới 100: OK")

# 12.7 các dạng đặc biệt n = 100
n = 100
special = {}
g = [[0] * n for _ in range(n)]
for i in range(n):
    g[i][0] = 1
    g[i][n - 1] = 1
special["hai cột hai mép (đảo dài)"] = (g, n - 2)
g = [[0] * n for _ in range(n)]
g[0][0] = 1
g[n - 1][n - 1] = 1
special["hai ô hai góc đối"] = (g, 2 * (n - 1) - 1)
g = [[0] * n for _ in range(n)]
for i in range(n):
    for j in range(n):
        if i == 0 or j == 0 or i == n - 1 or j == n - 1:
            g[i][j] = 1
g[n // 2][n // 2] = 1
special["khung ngoài và một ô ở tâm"] = (g, min(n // 2, n - 1 - n // 2) - 1)
for name, (g, expected) in special.items():
    assert len(components(g)) == 2, name
    for fname, fn in solutions_list().items():
        assert fn(g) == expected, (name, fname, fn(g), expected)
    print(f"  {name:30} đáp án {expected}")
print("12.7 dạng đặc biệt n = 100: OK")

12.4 ngẫu nhiên n <= 4 (400 lưới, gồm oracle tập con): OK


12.5 ngẫu nhiên n tới 12 (200 lưới): OK


12.6 hai hình chữ nhật, n tới 100: OK
  hai cột hai mép (đảo dài)      đáp án 98
  hai ô hai góc đối              đáp án 197
  khung ngoài và một ô ở tâm     đáp án 48
12.7 dạng đặc biệt n = 100: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — BFS chỉ từ **một** ô của đảo thứ nhất

Phải đưa **mọi ô** của đảo vào hàng đợi ban đầu; BFS từ một ô bỏ qua các ô đảo gần đảo kia hơn.

In [22]:
g = counters["BFS chỉ từ một ô"]
print("n =", len(g), "| sai ->", wrong_single_source(g), "| đúng ->", Solution().shortestBridge([r[:] for r in g]))
assert wrong_single_source(g) != Solution().shortestBridge([r[:] for r in g])

n = 4 | sai -> 2 | đúng -> 1


Vì sao sai: khi chỉ xuất phát từ một ô thì cả việc đi qua phần còn lại của đảo cũng bị tính là một bước. **Đúng:** hàng đợi ban đầu chứa mọi ô của đảo (khoảng cách `0`).

### Lỗi 2 — Đếm lệch `1`

Đáp án là số ô **nước** cần lật; nếu trả về số lớp tính cả bước vào ô đất của đảo kia sẽ lớn hơn `1`.

In [23]:
g = counters["đếm lệch 1"]
print("n =", len(g), "| sai ->", wrong_off_by_one(g), "| đúng ->", Solution().shortestBridge([r[:] for r in g]))
assert wrong_off_by_one(g) == Solution().shortestBridge([r[:] for r in g]) + 1
print("ví dụ 1: [[0,1],[1,0]] ->", wrong_off_by_one(examples[0][0]), "(sai) so với", Solution().shortestBridge([r[:] for r in examples[0][0]]))

n = 4 | sai -> 2 | đúng -> 1
ví dụ 1: [[0,1],[1,0]] -> 2 (sai) so với 1


Vì sao sai: khoảng cách (số cạnh) giữa hai đảo là số ô nước **cộng một**. **Đúng:** trả `steps` đúng thời điểm chạm (hoặc `khoảng cách - 1`).

### Lỗi 3 — Cho phép đi chéo

Đề dùng liên thông 4 hướng; đi chéo (8 hướng) cho đáp án nhỏ hơn.

In [24]:
g = counters["cho phép đi chéo"]
print("n =", len(g), "| sai ->", wrong_diagonal(g), "| đúng ->", Solution().shortestBridge([r[:] for r in g]))
assert wrong_diagonal(g) != Solution().shortestBridge([r[:] for r in g])
print("ví dụ 1:", wrong_diagonal(examples[0][0]), "(sai: hai đảo chéo đã tính là chạm nhau) so với 1")

n = 4 | sai -> 0 | đúng -> 1
ví dụ 1: 0 (sai: hai đảo chéo đã tính là chạm nhau) so với 1


Vì sao sai: hai ô chéo nhau **không** kề nhau theo 4 hướng nên chưa nối được. **Đúng:** chỉ 4 hướng.

### Lỗi 4 — Lấy khoảng cách của một cặp ô tùy ý

Khoảng cách cần là **nhỏ nhất** trên mọi cặp ô (hoặc BFS đa nguồn), không phải của hai ô đầu tiên.

In [25]:
g = counters["Manhattan của cặp tuỳ ý"]
print("n =", len(g), "| sai ->", wrong_arbitrary_pair(g), "| đúng ->", Solution().shortestBridge([r[:] for r in g]))
assert wrong_arbitrary_pair(g) != Solution().shortestBridge([r[:] for r in g])

n = 4 | sai -> 2 | đúng -> 1


Vì sao sai: hai ô được chọn có thể ở xa nhau trong khi một cặp khác ở gần. **Đúng:** nhỏ nhất theo mọi cặp (hoặc BFS đa nguồn `O(n^2)`).

### Lỗi 5 — DFS đệ quy để tìm đảo

Một đảo có thể có tới `~ 10^4` ô, nên DFS đệ quy vượt giới hạn mặc định `1000` của Python.

In [26]:
def dfs_recursive_island(grid):
    n = len(grid)
    seen = set()

    def dfs(x, y):
        seen.add((x, y))
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and grid[nx][ny] == 1 and (nx, ny) not in seen:
                dfs(nx, ny)

    start = next((i, j) for i in range(n) for j in range(n) if grid[i][j] == 1)
    dfs(*start)
    return len(seen)

n = 60
snake = [[0] * n for _ in range(n)]
for i in range(n):
    snake[i][0] = 1                                   # một cột dài 60 ô: chiều sâu đệ quy 60 (OK)
big_blob = [[1] * 50 + [0] * 10 for _ in range(60)]  # một khối 3000 ô: chiều sâu đệ quy lớn
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    dfs_recursive_island(big_blob)
    print("không lỗi?")
except RecursionError as e:
    print("RecursionError khi DFS đệ quy trên đảo 3000 ô:", e)
finally:
    sys.setrecursionlimit(old)
print("bản dùng ngăn xếp tường minh:", len(components(big_blob)[0]), "ô")

RecursionError khi DFS đệ quy trên đảo 3000 ô: maximum recursion depth exceeded
bản dùng ngăn xếp tường minh: 3000 ô


Vì sao sai: mỗi ô là một khung đệ quy. **Đúng:** DFS bằng ngăn xếp tường minh (hoặc BFS).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "nối hai thành phần liên thông bằng ít ô nhất" ⇒ **tìm một thành phần** (DFS) rồi **BFS đa nguồn** theo lớp tới thành phần kia.
- **BFS đa nguồn:** đưa **mọi** nguồn vào hàng đợi từ đầu; xử lý theo lớp `for _ in range(len(q))`.
- **Đếm đúng:** số ô nước = khoảng cách - 1; chạm đảo kia ở lớp thứ `steps` thì trả `steps`.
- Đánh dấu đã thăm bằng giá trị mới (`2`) trong lưới để phân biệt hai đảo.
- Cùng họ: Number of Islands (200), Max Area of Island (695), As Far from Land as Possible (1162), Rotting Oranges (994), 01 Matrix (542), Making A Large Island (827).
- Test nên có: hai ô chéo nhau, đảo lồng nhau, hai góc đối, `n = 100`.

## 15. Biến thể

1. **Trả về chính cầu** (danh sách ô cần lật; nhiều đáp án): bài tập 5.
2. **Nối `k` đảo** (đồ thị đầy đủ giữa các đảo): quy về cây khung nhỏ nhất, khó hơn.
3. **Cầu không được đi qua ô bị cấm:** đổi điều kiện mở rộng BFS.
4. Bài liên quan: Making A Large Island (827), Minimum Number of Days to Disconnect Island (1568).

Dưới đây là biến thể 1: truy vết một cầu ngắn nhất bằng BFS đa nguồn có `parent`, và hàm kiểm tra cầu hợp lệ.

In [27]:
def shortest_bridge_cells(grid: List[List[int]]) -> List[Tuple[int, int]]:
    g = [row[:] for row in grid]
    n = len(g)
    A = components(grid)[0]
    for x, y in A:
        g[x][y] = 2
    parent = {c: None for c in A}
    q = deque(A)
    while q:
        for _ in range(len(q)):
            x, y = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n:
                    if g[nx][ny] == 1:
                        path, cur = [], (x, y)
                        while parent[cur] is not None:        # bỏ ô của đảo A (parent None)
                            path.append(cur)
                            cur = parent[cur]
                        return path[::-1]
                    if g[nx][ny] == 0:
                        g[nx][ny] = 2
                        parent[(nx, ny)] = (x, y)
                        q.append((nx, ny))
    return []

def check_bridge(grid: List[List[int]], cells) -> bool:
    n = len(grid)
    if len(set(cells)) != len(cells) or any(grid[x][y] != 0 for x, y in cells):
        return False
    g = [row[:] for row in grid]
    for x, y in cells:
        g[x][y] = 1
    return len(components(g)) == 1 and len(cells) == bridge_manhattan(grid)

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(2, 8)
    g = random_two_island_grid(rng, n, rng.choice([0.1, 0.2, 0.3]))
    cells = shortest_bridge_cells(g)
    assert check_bridge(g, cells), (g, cells)
print("ví dụ 2: các ô cần lật", shortest_bridge_cells(examples[1][0]))

ví dụ 2: các ô cần lật [(1, 1), (2, 1)]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Manhattan mọi cặp | Khoảng cách nhỏ nhất - 1 | `O(\|A\| \|B\|)` | `O(n^2)` | Oracle độc lập |
| 2. DFS + BFS đa nguồn | Lan theo lớp từ đảo 1 | `O(n^2)` | `O(n^2)` | **Cách nộp chính** |
| 3. Hai BFS | `min(dA + dB) - 1` | `O(n^2)` | `O(n^2)` | Không cần phân biệt thứ tự |
| 4. BFS `0-1` | Đất giá 0, nước giá 1 | `O(n^2)` | `O(n^2)` | Không cần liệt kê đảo |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(components([[0, 1], [1, 0]]))
print(list(pairwise((-1, 0, 1, 0, -1))))
print(abs(0 - 1) + abs(1 - 0) - 1)
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Lưới `4 x 4` chỉ có hai ô đất ở hai góc đối: `grid[0][0] = 1` và `grid[3][3] = 1` (còn lại là nước). Số ô nước ít nhất cần lật?

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS đa nguồn theo lớp (cho trước `island` là danh sách các ô của đảo thứ nhất đã được đánh dấu `2` trong `g`).

In [30]:
def exercise_3_bfs(g: List[List[int]], island) -> int:
    n = len(g)
    q = deque(island)
    steps = 0
    while q:
        for _ in range(len(q)):
            x, y = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n:
                    # TODO 1: nếu g[nx][ny] == 1 (đảo kia) thì trả steps
                    # TODO 2: nếu g[nx][ny] == 0 thì đánh dấu 2 và thêm vào hàng đợi
                    pass
        # TODO 3: tăng steps sau mỗi lớp
    return -1


# for grid, expected in examples:
#     g = [r[:] for r in grid]
#     A = components(grid)[0]
#     for x, y in A:
#         g[x][y] = 2
#     assert exercise_3_bfs(g, A) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_01bfs(grid)`: BFS `0-1` từ một ô đất, trả về khoảng cách dương nhỏ nhất tới một ô đất.

In [31]:
def exercise_4_01bfs(grid: List[List[int]]) -> int:
    # TODO
    pass


# for g, expected in examples:
#     assert exercise_4_01bfs(g) == expected

### Bài 5 — Mini challenge: trả về các ô cần lật (khó)

Viết `exercise_5_bridge(grid)` trả về **danh sách các ô** `(hàng, cột)` cần lật (đúng bằng đáp án tối thiểu) để nối hai đảo. Có nhiều đáp án; ta kiểm tra bằng hàm kiểm tra (`check_bridge`: các ô đều là nước, sau khi lật còn đúng một đảo, và số ô bằng đáp án tối thiểu).

```python
exercise_5_bridge([[0, 1], [1, 0]])             -> [(0, 0)] hoặc [(1, 1)]
exercise_5_bridge([[0, 1, 0], [0, 0, 0], [0, 0, 1]])  -> hai ô, ví dụ [(0, 2), (1, 2)]
```

In [32]:
def exercise_5_bridge(grid: List[List[int]]) -> List[Tuple[int, int]]:
    # TODO
    pass


# for g, expected in examples:
#     cells = exercise_5_bridge(g)
#     assert check_bridge(g, cells) and len(cells) == expected

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `components` trả về danh sách các đảo, mỗi đảo là danh sách ô.
- Mức 2: lưới `[[0,1],[1,0]]` có hai đảo một ô: `(0,1)` và `(1,0)`; Manhattan giữa chúng là `2`.
- Mức 3: `[[(0, 1)], [(1, 0)]]`, `[(-1, 0), (0, 1), (1, 0), (0, -1)]`, `1`.

**Bài 2**
- Mức 1: khoảng cách Manhattan giữa `(0,0)` và `(3,3)` là `6`.
- Mức 2: số ô nước giữa là `6 - 1`.
- Mức 3: `5`.

**Bài 3**
- Mức 1: `if g[nx][ny] == 1: return steps`.
- Mức 2: `if g[nx][ny] == 0: g[nx][ny] = 2; q.append((nx, ny))`.
- Mức 3: `steps += 1` ở cuối mỗi lớp (sau vòng `for _ in range(len(q))`).

**Bài 4**
- Mức 1: nguồn là một ô đất bất kỳ; `dist` khởi tạo vô cùng.
- Mức 2: bước vào ô đất giá `0` (`appendleft`), vào ô nước giá `1` (`append`).
- Mức 3: đáp án `min` của `dist` trên các ô đất có `dist > 0`.

**Bài 5**
- Mức 1: BFS đa nguồn có `parent`.
- Mức 2: khi gặp đảo kia từ ô `(x, y)`, truy ngược `parent`.
- Mức 3: bỏ các ô thuộc đảo thứ nhất (`parent is None`); đảo thứ tự danh sách.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(components([[0, 1], [1, 0]]))
print(list(pairwise((-1, 0, 1, 0, -1))))
print(abs(0 - 1) + abs(1 - 0) - 1)

assert components([[0, 1], [1, 0]]) == [[(0, 1)], [(1, 0)]]
assert list(pairwise((-1, 0, 1, 0, -1))) == [(-1, 0), (0, 1), (1, 0), (0, -1)]
assert abs(0 - 1) + abs(1 - 0) - 1 == 1

[[(0, 1)], [(1, 0)]]
[(-1, 0), (0, 1), (1, 0), (0, -1)]
1


### Đáp án Bài 2

Khoảng cách Manhattan `|0 - 3| + |0 - 3| = 6`; số ô nước giữa `6 - 1 = 5`.

In [34]:
hand_answer_solution = 5
g = [[0] * 4 for _ in range(4)]
g[0][0] = 1
g[3][3] = 1
assert Solution().shortestBridge([r[:] for r in g]) == hand_answer_solution == bridge_manhattan(g)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_bfs_solution(g: List[List[int]], island) -> int:
    n = len(g)
    q = deque(island)
    steps = 0
    while q:
        for _ in range(len(q)):
            x, y = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n:
                    if g[nx][ny] == 1:
                        return steps
                    if g[nx][ny] == 0:
                        g[nx][ny] = 2
                        q.append((nx, ny))
        steps += 1
    return -1

def run_exercise_3(grid):
    g = [r[:] for r in grid]
    A = components(grid)[0]
    for x, y in A:
        g[x][y] = 2
    return exercise_3_bfs_solution(g, A)

for grid, expected in examples:
    assert run_exercise_3(grid) == expected
rng = random.Random(3)
for _ in range(200):
    g = random_two_island_grid(rng, rng.randint(2, 8), rng.choice([0.1, 0.2, 0.3]))
    assert run_exercise_3(g) == bridge_manhattan(g)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_01bfs_solution(grid: List[List[int]]) -> int:
    n = len(grid)
    start = next((i, j) for i in range(n) for j in range(n) if grid[i][j] == 1)
    INF = 10 ** 9
    dist = [[INF] * n for _ in range(n)]
    dist[start[0]][start[1]] = 0
    dq = deque([start])
    while dq:
        x, y = dq.popleft()
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                w = 0 if grid[nx][ny] == 1 else 1
                if dist[x][y] + w < dist[nx][ny]:
                    dist[nx][ny] = dist[x][y] + w
                    (dq.appendleft if w == 0 else dq.append)((nx, ny))
    return min(dist[i][j] for i in range(n) for j in range(n) if grid[i][j] == 1 and dist[i][j] > 0)

for g, expected in examples:
    assert exercise_4_01bfs_solution(g) == expected
rng = random.Random(4)
for _ in range(200):
    g = random_two_island_grid(rng, rng.randint(2, 8), rng.choice([0.1, 0.2, 0.3]))
    assert exercise_4_01bfs_solution(g) == bridge_manhattan(g)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_bridge_solution(grid: List[List[int]]) -> List[Tuple[int, int]]:
    return shortest_bridge_cells(grid)

for g, expected in examples:
    cells = exercise_5_bridge_solution(g)
    assert check_bridge(g, cells) and len(cells) == expected, (g, cells)
rng = random.Random(5)
for _ in range(200):
    g = random_two_island_grid(rng, rng.randint(2, 8), rng.choice([0.1, 0.2, 0.3]))
    assert check_bridge(g, exercise_5_bridge_solution(g)), g
print("Bài 5 OK", exercise_5_bridge_solution(examples[1][0]))

Bài 5 OK [(1, 1), (2, 1)]


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DFS lấy đảo thứ nhất + BFS đa nguồn theo lớp)

```python
class Solution:
    def shortestBridge(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        dirs = (-1, 0, 1, 0, -1)
        start = next((i, j) for i in range(n) for j in range(n) if grid[i][j])
        island, stack = [], [start]
        grid[start[0]][start[1]] = 2                          # 2 = đảo thứ nhất (đã thăm)
        while stack:                                          # DFS bằng ngăn xếp tường minh
            x, y = stack.pop()
            island.append((x, y))
            for a, b in zip(dirs, dirs[1:]):
                nx, ny = x + a, y + b
                if 0 <= nx < n and 0 <= ny < n and grid[nx][ny] == 1:
                    grid[nx][ny] = 2
                    stack.append((nx, ny))
        q, steps = deque(island), 0                           # BFS đa nguồn: MỌI ô của đảo
        while q:
            for _ in range(len(q)):                           # đúng một lớp
                x, y = q.popleft()
                for a, b in zip(dirs, dirs[1:]):
                    nx, ny = x + a, y + b
                    if 0 <= nx < n and 0 <= ny < n:
                        if grid[nx][ny] == 1:                 # chạm đảo kia
                            return steps
                        if grid[nx][ny] == 0:
                            grid[nx][ny] = 2
                            q.append((nx, ny))
            steps += 1
        return -1
```

Công thức nhớ nhanh: đáp án = số lớp BFS đa nguồn (từ mọi ô đảo 1) cho tới khi kề đảo 2 = `khoảng cách - 1`.

### Checklist trước khi submit

- [ ] Hàng đợi ban đầu chứa **mọi** ô của đảo thứ nhất (không chỉ một ô)?
- [ ] Liên thông **4 hướng** (không đi chéo)?
- [ ] Đếm đúng: trả `steps` khi gặp đảo kia (không cộng thêm `1`)?
- [ ] Dùng ngăn xếp tường minh cho DFS (không đệ quy sâu)?
- [ ] Đã đánh dấu đã thăm (`2`) cho cả đảo và ô nước đã lan?
- [ ] Đã thử: hai ô chéo nhau, đảo lồng nhau, hai góc đối, `n = 100`?
- [ ] Tên method đúng `shortestBridge(self, grid)` và trả `int`?